# Compare spike types between episode parts

This notebook analyzes the relative occurrence of AIS-AP and RA-AP in the first and last part of a given episode.

## Initialization - don't change

In [17]:
import sys
sys.path.append('..')

from pathlib import Path

import pandas as pd

from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

data_params = InputDataParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [18]:
palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Input-output settings

Base directory paths
 - `MAT_DIR` - base dir of the raw **mat** files, grouped into subfolders
 - `PROCESSED_DIR` - base dir of the preprocessed data and results

In [19]:
MAT_DIR = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
PROCESSED_DIR = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed'

In [20]:
input_param = {
    'mode': 'table',
    'table_path': f'{PROCESSED_DIR}\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

In [21]:
data_params.raw_data.path= MAT_DIR
data_params.raw_data.extension='mat'
data_params.spike_features.path = f'{PROCESSED_DIR}\\SpikeFeatures'
data_params.spike_features.extension = 'csv'

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [22]:
output_path = Path(PROCESSED_DIR) / 'Episode_Part_Analysis'
result_path = output_path / '01_results'
individual_path = result_path / 'individual'
figure_path = output_path / '02_figures'
statistics_path = output_path / '03_statistics'

## Analysis parameters

Name of the spike types

In [23]:
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

List of the groups in the order of display.

In [24]:
group_list = [
    'FSI_L2-3',
    'RSI_L2-3',
    'PYR_L2-3',
    'FSI_L4-6',
    'RSI_L4-6',
    'PYR_L4-6'
]

 - `episode_name`: The name of the episode to analyze

In [25]:
episode_name = 'sw'

 - `episode_part_count`: the number of parts to divide each episode

In [26]:
episode_part_count = 2

## Plot parameters

 - `y_limit`: y axis limit for the boxplots

In [27]:
y_limit = (0, 1)

## Run analysis

Collect input files according to the table.

In [28]:
input_files = collect_paths_from_table(data_params, input_param)
n_cells = len(input_files)

In [29]:
from workflows.episode_part_analysis import (
    process_cell,
    aggregate_cell,
    flatten_aggregate_df
)

results = [
    process_cell(
        input_file,
        episode_name=episode_name,
        episode_part_count=episode_part_count,
        spike_types=spike_types,
        individual_path=individual_path
    ) for input_file in input_files
]

In [30]:
cell_aggregates = [
    aggregate_cell(
        cell_data,
        episode_part_count=episode_part_count,
        spike_types=spike_types
    ) for cell_data in results
]
cell_aggregate_df = pd.concat(cell_aggregates)
result_path.mkdir(exist_ok=True, parents=True)
cell_aggregate_df.to_excel(
    f'{result_path}/cell_mean_{episode_name}_partition_values.xlsx',
    index=False
)

In [31]:
import numpy as np
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

flatten_frame = flatten_aggregate_df(cell_aggregate_df, spike_types)

stat_results = []
for part_i in [0, episode_part_count-1]:
    p_raws = []
    groups_medians  = []
    for group_name in group_list:
        group_values = []
        for spike_type in ['AIS-AP', 'RA-AP']:
            group_values.append(flatten_frame[
                (flatten_frame['part_id'] == part_i) &
                (flatten_frame['group'] == group_name) &
                (flatten_frame['spike_type'] == spike_type)
            ]['value'].to_numpy())
        _, p = ranksums(group_values[0], group_values[1])
        groups_medians.append(
            (np.median(group_values[0]), np.median(group_values[1]))
        )
        p_raws.append(p)

    _, p_adjs, _, _ = multipletests(p_raws, method='holm')

    for group_name, group_medians, p_raw, p_adj in zip(group_list, groups_medians, p_raws, p_adjs):
        stat_results.append({
            'part_id': part_i,
            'group': group_name,
            'AIS-AP_median': group_medians[0],
            'RA-AP_median': group_medians[1],
            'p_raw': p_raw,
            'p_adj': p_adj
        })
stat_result_df = pd.DataFrame(stat_results)
statistics_path.mkdir(exist_ok=True, parents=True)
stat_result_df.to_excel(
    f'{statistics_path}/spike_type_in_{episode_name}_part_comparison.xlsx',
    index=False
)

In [32]:
import matplotlib.pyplot as plt
import seaborn as sns

from workflows.spike_ac_analysis import save_figure

figure_path.mkdir(exist_ok=True, parents=True)

for part_i in [0, episode_part_count-1]:
    fig, axes = plt.subplots(1, len(group_list), figsize=(20, 5))
    for col_i, group_name in enumerate(group_list):
        ax = axes[col_i]
        group_part_data = flatten_frame[
            (flatten_frame['group'] == group_name) &
            (flatten_frame['part_id'] == part_i) &
            (flatten_frame['spike_type'].isin(['AIS-AP', 'RA-AP']))
        ]

        sns.boxplot(
            data=group_part_data,
            x='spike_type',
            y='value',
            width=0.5,
            fill=False,
            color='gray',
            fliersize=0,
            ax=ax
        )
        for spike_type in ['AIS-AP', 'RA-AP']:
            color = palette['spike_type']['RA-AP'] if spike_type == 'RA-AP' else palette['cell_type'][group_name]
            group_part_type_data = group_part_data[group_part_data['spike_type'] ==  spike_type]
            sns.stripplot(
                data=group_part_type_data,
                x='spike_type',
                y='value',
                color=color,
                ax=ax
            )

        ax.set_title(f'{group_name}')
        ax.set_ylim(y_limit)
    plt.suptitle(f'{episode_name.capitalize()} {part_i + 1}. part')
    save_figure(
        figure_path,
        fig,
        f'spike_type_in_{episode_name}_parts_{part_i + 1}_boxplot',
        to_svg=True
    )
    plt.close(fig)


In [33]:
import numpy as np
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

flatten_frame = flatten_aggregate_df(cell_aggregate_df, spike_types)

stat_results = []
for spike_type in ['AIS-AP', 'RA-AP']:
    p_raws = []
    groups_medians  = []
    for group_name in group_list:
        group_values = []
        for part_i in [0, episode_part_count-1]:
            group_values.append(flatten_frame[
                (flatten_frame['part_id'] == part_i) &
                (flatten_frame['group'] == group_name) &
                (flatten_frame['spike_type'] == spike_type)
            ]['value'].to_numpy())
        _, p = ranksums(group_values[0], group_values[1])
        groups_medians.append(
            (np.median(group_values[0]), np.median(group_values[1]))
        )
        p_raws.append(p)

    _, p_adjs, _, _ = multipletests(p_raws, method='holm')

    for group_name, group_medians, p_raw, p_adj in zip(group_list, groups_medians, p_raws, p_adjs):
        stat_results.append({
            'spike_type': spike_type,
            'group': group_name,
            f'median_part_1': group_medians[0],
            f'median_part_{episode_part_count}': group_medians[1],
            'p_raw': p_raw,
            'p_adj': p_adj
        })
stat_result_df = pd.DataFrame(stat_results)
statistics_path.mkdir(exist_ok=True, parents=True)
stat_result_df.to_excel(
    f'{statistics_path}/{episode_name}_first_vs_last_part_spiketype_comparison.xlsx',
    index=False
)

In [37]:
import matplotlib.pyplot as plt
import seaborn as sns

from workflows.spike_ac_analysis import save_figure

figure_path.mkdir(exist_ok=True, parents=True)

for spike_type in ['AIS-AP', 'RA-AP']:
    fig, axes = plt.subplots(1, len(group_list), figsize=(20, 5))
    for col_i, group_name in enumerate(group_list):
        ax = axes[col_i]
        group_part_data = flatten_frame[
            (flatten_frame['group'] == group_name) &
            (flatten_frame['part_id'].isin([0, episode_part_count-1])) &
            (flatten_frame['spike_type'] == spike_type)
        ]

        sns.boxplot(
            data=group_part_data,
            x='part_id',
            y='value',
            width=0.5,
            fill=False,
            color='gray',
            fliersize=0,
            ax=ax
        )
        color = palette['spike_type']['RA-AP'] if spike_type == 'RA-AP' else palette['cell_type'][group_name]
        x_labels = []
        for part_id in [0, episode_part_count-1]:
            group_part_type_data = group_part_data[group_part_data['part_id'] == part_id]
            sns.stripplot(
                data=group_part_type_data,
                x='part_id',
                y='value',
                color=color,
                ax=ax
            )
            x_labels.append(f'Part {part_id + 1}')
        ax.set_xticks(ticks=[0, 1],labels=x_labels)
        ax.set_title(f'{group_name}')
        ax.set_ylim(y_limit)
    plt.suptitle(f'{spike_type} {episode_name.capitalize()} first vs last part')
    save_figure(
        figure_path,
        fig,
        f'{episode_name}_first_vs_last_parts_{spike_type}_boxplot',
        to_svg=True
    )
    plt.close(fig)
